# Demo 2 — Paired t-test equivalent

This demo reuses the `sleep` dataset (R base; Cushny & Peebles 1905 / Student
1908), but now takes account of the fact that both drugs were measured on the
same 10 patients. It asks the same question as demo 1 — does sleep gain differ
between the drugs — while respecting the within-patient pairing.

A random intercept per patient absorbs each person's baseline sleep tendency:

    extra ~ group + (1 | ID)

This is equivalent to a paired t-test (degrees of freedom n − 1), and comparing
its p-value with demo 1 shows how accounting for the pairing increases power.

## Setup

In [ ]:
import os
try:
    notebook_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    notebook_dir = os.getcwd()  # JupyterLab sets CWD to the notebook directory
os.chdir(notebook_dir)

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## Options

Adding `options.id = 'ID'` introduces a random intercept per patient, making this a linear mixed model equivalent to a paired t-test.

In [ ]:
options = KbstatOptions()
options.in_file  = os.path.join(notebook_dir, '../data/sleep.csv')
options.out_dir  = ''   # empty: show results inline only; set a folder to also save them
options.y        = 'extra'
options.y_units  = 'h'
options.x        = 'group'
options.id       = 'ID'
options.rename   = 'extra -> ExtraSleep; group -> DrugGroup'

## Model fitting

`fit()` estimates the model parameters via restricted maximum likelihood (REML).

In [ ]:
kb = Kbstat(options)
kb.fit()
print(f'Formula : {kb._build_formula()}')
print(f'AIC     : {kb.AIC:.3f}')
print(f'BIC     : {kb.BIC:.3f}')
print(f'logLik  : {kb.logLik:.3f}')

## ANOVA table (Type III)

In [ ]:
kb.anova()
kb.anova_table

## Post-hoc pairwise comparisons

One comparison. With pairing, power increases — compare the p-value with Demo 1.

In [ ]:
kb.posthoc()
kb.posthoc_table

## Data plot

Violin + jitter scatter with EMM and 95 % CI overlaid.

> An interactive version with hover tooltips is written as an HTML file when `out_dir` is set.

In [ ]:
kb.plot_data()
plt.show()

## Diagnostic plots

Six panels checking model assumptions.

See [STATISTICAL_NOTES.md](../../STATISTICAL_NOTES.md) for interpretation guidance.

In [ ]:
kb.plot_diagnostics()
plt.show()

## Save results

Everything above is shown inline. To also write all tables, the summary, and the figures (PDF/PNG and interactive HTML) to disk, uncomment the two lines in the cell below — they set `out_dir` and call `save()`.

In [ ]:
# To also write the tables, summary, and figures to disk, uncomment both lines:
# options.out_dir = os.path.join(notebook_dir, 'results/demo_02_paired')
# kb.save()

## Interpretation

- The random intercept absorbs between-subject baseline differences.
- The post-hoc p-value should be lower than in Demo 1, reflecting increased power from pairing.
- For balanced data with one random intercept, this model is algebraically identical to a paired t-test (Satterthwaite df = 9 exactly).